In [3]:
import csv
import sqlite3

TRACES = "traces-td1.csv"

con = sqlite3.connect(":memory:")
con.execute("""
CREATE TABLE traces (
    case_id TEXT, provider TEXT, model_requested TEXT,
    value TEXT, reference TEXT, verdict TEXT, attempts INTEGER
)
""")

csv.field_size_limit(10_000_000)
with open(TRACES, encoding="utf-8-sig", newline="") as f:
    for r in csv.DictReader(f):
        con.execute("INSERT INTO traces VALUES (?, ?, ?, ?, ?, ?, ?)",
                    (r["case_id"], r["provider"], r["model_requested"],
                     r["value"], r["reference"], r["verdict"], int(r["attempts"])))

def run(sql):
    """Run the query and print each row."""
    rows = con.execute(sql).fetchall()
    for row in rows:
        print(row)
    print(f"-> {len(rows)} row(s)")
    return rows

We keep only the **last attempt** of each model on each question, because that is the one that counts in the vote.

In [2]:
con.execute("""
CREATE VIEW latest_attempts AS
SELECT t.*
FROM traces t
JOIN (SELECT model_requested, case_id, MAX(attempts) AS last_attempt
      FROM traces
      GROUP BY model_requested, case_id) m
  ON  t.model_requested = m.model_requested
  AND t.case_id = m.case_id
  AND t.attempts = m.last_attempt
""");

In [3]:
shared_wrong_value = run("""
SELECT case_id, value, reference,
       COUNT(DISTINCT model_requested) AS nb_models,
       GROUP_CONCAT(model_requested, ' + ') AS models
FROM latest_attempts
WHERE verdict = 'incorrect'
GROUP BY case_id, value
HAVING COUNT(DISTINCT model_requested) >= 2
ORDER BY case_id;
""")

-> 0 row(s)


## Part 1: the chosen case and the votes


We looked for a question where at least 2 models give the same wrong value.
the query above returns 0 rows. Our models are sometimes wrong, but never with the same value on the same question.

 in this situation, the assignment tells us to use the teacher's case, DEMO-SQL-C11

## 2. Case C11

Since we have no shared wrong value in our traces, we use the teacher's case:
  DEMO-SQL-C11. It comes from a pilot run in September 2026, so it's not from our
  batch.

  The question is: what is the revenue from customers who are exactly 25 years
  old on 2026-12-01?

  In this pilot, gpt-oss-20b and north-mini-code:free both answered 0. The right
  answer is 446690 (from our reference-td1.csv), so they agree on a wrong value.

  In our own TD 1 run, Cohere also answered 0, but 20B answered 322160. Let's
  look at what our 5 models answered on C11, we use these answers for the votes.

In [4]:
c11_answers = run("""
SELECT model_requested, verdict, value
FROM latest_attempts
WHERE case_id = 'C11'
ORDER BY model_requested;
""")

('cohere/north-mini-code:free', 'incorrect', '0')
('dots-studio/dots-3-note-preview:free', 'correct', '446690')
('openai/gpt-oss-120b', 'correct', '446690')
('openai/gpt-oss-20b', 'incorrect', '322160')
('qwen/qwen3.8-27b', 'correct', '446690')
-> 5 row(s)


## 3. 2 of 3

The voters: gpt-oss-120b  north-mini-code:free and qwen3.8-27b. a value wins if it gets at least 2 votes.
 each model gives one vote to its answer. I group identical answers and count the votes.

In [5]:
vote_2_of_3 = run("""
SELECT value, COUNT(*) AS votes, GROUP_CONCAT(model_requested, ' + ') AS models
FROM latest_attempts
WHERE case_id = 'C11'
  AND verdict IN ('correct', 'incorrect')
  AND model_requested IN ('openai/gpt-oss-120b',
                          'cohere/north-mini-code:free',
                          'qwen/qwen3.8-27b')
GROUP BY value
ORDER BY votes DESC;
""")

('446690', 2, 'openai/gpt-oss-120b + qwen/qwen3.8-27b')
('0', 1, 'cohere/north-mini-code:free')
-> 2 row(s)


**Result:** 446690 gets 2 votes, so it wins. It is the correct value. Cohere's 0 has only 1 vote.

## 4. 3 of 5

 our 5 TD 1 models.a value wins if it gets at least 3 votes

In [6]:
vote_3_of_5 = run("""
SELECT value, COUNT(*) AS votes, GROUP_CONCAT(model_requested, ' + ') AS models
FROM latest_attempts
WHERE case_id = 'C11'
  AND verdict IN ('correct', 'incorrect')
GROUP BY value
ORDER BY votes DESC;
""")

('446690', 3, 'openai/gpt-oss-120b + qwen/qwen3.8-27b + dots-studio/dots-3-note-preview:free')
('322160', 1, 'openai/gpt-oss-20b')
('0', 1, 'cohere/north-mini-code:free')
-> 3 row(s)


**Result:** 446690 gets 3 votes, so it wins. It is the correct value again.

Let's see if the shared value wins :

In [9]:
# 120B, Qwen and Dots keep our answer. 20B and Cohere answer 0.
votes_5_models = [446690, 446690, 446690, 0, 0]
print("3-of-5 vote: 446690 has", votes_5_models.count(446690), "votes, 0 has", votes_5_models.count(0), "votes")

3-of-5 vote: 446690 has 3 votes, 0 has 2 votes


**Result:**

- With 5 models, 0 has only 2 votes. It needs 3, so it loses.
- In our fixed 3-model panel, 20B is not present, so these two models cannot give 0 two votes.


## 5. Independent check in Python

We find the right customers and the right total directly from the CSV files.


1. We do 2026 minus the birth year;
2. if the birthday falls after December 1, the person has not had it yet, so we subtract 1.

In [8]:
DATA_FOLDER = "../esilv-student-2026-10-14/kit-td1/data/"

# Price of each product
prices = {}
with open(DATA_FOLDER + "product.csv", encoding="utf-8", newline="") as f:
    for row in csv.DictReader(f):
        prices[row["product_id"]] = int(row["price"])

# Customers who are exactly 25 years old on 2026-12-01
customer_revenue = {}
customer_birth_date = {}
with open(DATA_FOLDER + "customers.csv", encoding="utf-8", newline="") as f:
    for row in csv.DictReader(f):
        birth_date = row["birth_date"]
        age = 2026 - int(birth_date[:4])
        if birth_date[5:] > "12-01":
            age = age - 1
        if age == 25:
            customer_revenue[row["customer_id"]] = 0
            customer_birth_date[row["customer_id"]] = birth_date

# Add the amount of each order from these customers
with open(DATA_FOLDER + "orders.csv", encoding="utf-8", newline="") as f:
    for row in csv.DictReader(f):
        customer = row["customer_id"]
        if customer in customer_revenue:
            customer_revenue[customer] = customer_revenue[customer] + int(row["quantity_product"]) * prices[row["product_id"]]

# Display
total = 0
for customer in sorted(customer_revenue):
    print(customer, customer_birth_date[customer], customer_revenue[customer])
    total = total + customer_revenue[customer]

reference = int(con.execute("SELECT reference FROM traces WHERE case_id = 'C11'").fetchone()[0])
print("Number of customers:", len(customer_revenue))
print("Python total:", total)
print("Reference:", reference)
print("Identical:", total == reference)

C001 2001-09-25 135000
C002 2001-09-24 25920
C003 2001-09-26 47900
C006 2001-11-30 77450
C008 2001-05-15 99090
C009 2001-03-10 61330
Number of customers: 6
Python total: 446690
Reference: 446690
Identical: True


**Result:** Python finds 6 customers and a total of 446690, the same as our reference.

These 6 customers explain the whole gap with the wrong value 0: the two wrong queries did not count them.

# Part 2: 

## 1. Verifying the results on the database

In [11]:
DB = "data/boutique.sqlite"
con = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)

def run_db(sql):
    """Execute la requete et renvoie l'unique valeur revenue_cents."""
    rows = con.execute(sql).fetchall()
    return rows

results = {}

In [7]:
# M1 Query
sql = """
SELECT COALESCE(SUM(o.quantity_product * p.price), 0) AS revenue_cents
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN product p ON o.product_id = p.product_id
WHERE c.birth_date = '2001-12-01';
"""

results["M1"] =  run_db(sql)[0][0]

In [8]:
# M2 Query
sql = """
SELECT COALESCE(SUM(o.quantity_product * p.price), 0) AS revenue_cents
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN product p ON o.product_id = p.product_id
WHERE CAST(strftime('%Y-%m-%d', '2026-12-01') AS TEXT) >= c.birth_date
AND CAST(strftime('%Y-%m-%d', '2026-12-01', '+1 year') AS TEXT) <= c.birth_date;
"""

results["M2"] = run_db(sql)[0][0]

In [9]:
print("M1:", results["M1"])
print("M2:", results["M2"])

M1: 0
M2: 0


We see that the results are wrong in both case, giving off a revenue of 0 cents for two dinstincs reasons.

## 2. Identifying the mistakes

* For the **M1** query, we see that it only checks for one birth date with `c.birth_date = '2001-12-01';`. So the only customers taken into this calculations are customers whose birth date is exactly the 1st of december 2001, in this case there are none.
* For the **M2** query, looking at the `WHERE` conditions reveals 2 mistakes. First, the interval of birth dates that must be validated is impossible (ealier than December 2026 and also later than December 2027). Even with fixing the interval to only pick birth dates from a single selected year, we need to make it so the year selected in between December 2000 and December 2001 in order to have customers whom are 25 years old (at the time of 2026). Since no customers can validate these impossible conditions, the revenue returned is 0

In [13]:
sql = """
SELECT 
    c.customer_id,
    c.birth_date,
    COALESCE(SUM(o.quantity_product * p.price), 0) AS revenue_cents
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
LEFT JOIN product p ON o.product_id = p.product_id
WHERE c.birth_date > date('2026-12-01', '-26 years')
  AND c.birth_date <= date('2026-12-01', '-25 years')
GROUP BY c.customer_id, c.birth_date
ORDER BY c.customer_id;"""

customers = run_db(sql)
for i in range(len(customers)):
    print("customerID : " + customers[i][0] + ", birth_date : " + customers[i][1] + ", revenue : " + str(customers[i][2]))

customerID : C001, birth_date : 2001-09-25, revenue : 135000
customerID : C002, birth_date : 2001-09-24, revenue : 25920
customerID : C003, birth_date : 2001-09-26, revenue : 47900
customerID : C006, birth_date : 2001-11-30, revenue : 77450
customerID : C008, birth_date : 2001-05-15, revenue : 99090
customerID : C009, birth_date : 2001-03-10, revenue : 61330


Same results we found in python

## DEMO-VOTE (invented example)

  This example comes from the assignment. It is not a model answer and it
  does not use the shop database. With `>`, the sale on 2026-01-01 is
  missed, so the answer is 0. With `>=`, it is counted, so the answer is
  100. With 2 votes needed, 0 wins even though it is wrong.

In [14]:
demo = sqlite3.connect(":memory:")
demo.execute("CREATE TABLE sales (sale_date TEXT, amount_cents INTEGER)")
demo.execute("INSERT INTO sales VALUES ('2026-01-01', 100)")

wrong = demo.execute("""
SELECT COALESCE(SUM(amount_cents), 0)
FROM sales
WHERE sale_date > '2026-01-01' AND sale_date < '2026-01-02'
""").fetchone()[0]

right = demo.execute("""
SELECT COALESCE(SUM(amount_cents), 0)
FROM sales
WHERE sale_date >= '2026-01-01' AND sale_date < '2026-01-02'
""").fetchone()[0]

votes = [wrong, wrong, right]
print("Votes:", votes)
print(wrong, "has", votes.count(wrong), "votes")
print(right, "has", votes.count(right), "vote")

Votes: [0, 0, 100]
0 has 2 votes
100 has 1 vote
